In [2]:
import ee


# --- 用户配置 ---

# 1. GEE认证与初始化
ee.Authenticate()
# Initialize Earth Engine
ee.Initialize(
    project='ee-lbwnb331161',
    opt_url='https://earthengine-highvolume.googleapis.com'
)

# 2. 输入/输出参数
HEX_GRID_ASSET_ID = 'projects/ee-lbwnb331161/assets/mangrove_2020_500m_hex_grid'
COASTLINE_ASSET_ID = 'projects/ee-lbwnb331161/assets/S2Coast-2023_Polyline_diss'
TARGET_YEARS = [2000,2005,2010,2015,2020,2024]
OUTPUT_FOLDER = 'GEE_Mangrove_Data'
OUTPUT_FILENAME_PREFIX = 'mangrove_hexgrid_environmental_vars'
REDUCER_SCALE = 1000  # 网格分辨率为1km，因此使用1000m的缩放系数

# --- 数据集ID ---
ERA5_TEMP_ID = "ECMWF/ERA5_LAND/MONTHLY_AGGR"
CHIRPS_PRECIP_ID = "UCSB-CHG/CHIRPS/PENTAD"
MODIS_LST_ID = "MODIS/061/MOD11A2"
MODIS_SST_ID = "NASA/OCEANDATA/MODIS-Aqua/L3SMI"
JRC_WATER_ID = "JRC/GSW1_4/GlobalSurfaceWater"

print("配置加载完成。")

# --- 数据处理函数 ---

def get_static_variables(coastline_id, water_id, hex_grid):
    """
    计算不随年份变化的静态地理变量。
    - 离岸距离 (Distance to Coast)
    - 年均水体出现频率 (Water Occurrence)
    """
    print("正在计算静态变量 (离岸距离, 水体频率)...")
    
    # 获取网格的边界范围,扩展50km用于距离计算缓冲区
    bounds = hex_grid.geometry().bounds().buffer(50000)
    
    # 1. 使用 FeatureCollection 海岸线计算距离
    coastline = ee.FeatureCollection(coastline_id).filterBounds(bounds)
    
    # 将海岸线 FeatureCollection 转换为图像,用于距离计算
    # 创建一个空白图像,将海岸线栅格化
    coastline_image = ee.Image().byte().paint(
        featureCollection=coastline,
        color=1
    )
    
    # 计算到海岸线的距离
    dist_to_coast = coastline_image.fastDistanceTransform(
        neighborhood=256,
        units='meters',
        metric='squared_euclidean'
    ).sqrt().multiply(ee.Image.pixelArea().sqrt()).rename('dist_to_coast')

    # 2. 年均水体出现频率
    water_occurrence = ee.Image(water_id).select('occurrence').rename('water_occurrence')

    return dist_to_coast.addBands(water_occurrence)


def get_annual_variables(year, hex_grid):
    """
    为指定年份计算各项年度环境变量。
    """
    print(f"  - 开始处理 {year} 年的年度变量...")
    bounds = hex_grid.geometry().bounds().buffer(50000)
    start_date = ee.Date.fromYMD(year, 1, 1)
    end_date = ee.Date.fromYMD(year, 12, 31)

    # 1. 温度 (ERA5-Land)
    era5_collection = ee.ImageCollection(ERA5_TEMP_ID) \
        .filterDate(start_date, end_date) \
        .select('temperature_2m')

    # 温度单位从开尔文(K)转换为摄氏度(°C)
    def k_to_c(image):
        return image.subtract(273.15).copyProperties(image, ['system:time_start'])

    era5_celsius = era5_collection.map(k_to_c)
    
    mean_temp = era5_celsius.mean().rename('mean_annual_temp')
    min_temp = era5_celsius.min().rename('coldest_month_temp') # 最冷月均温

    # 2. 降水 (CHIRPS Pentad)
    chirps_collection = ee.ImageCollection(CHIRPS_PRECIP_ID) \
        .filterDate(start_date, end_date) \
        .select('precipitation')
    
    total_precip = chirps_collection.sum().rename('total_annual_precip')

    # 3. 白天地表温度 (MODIS LST)
    modis_lst_collection = ee.ImageCollection(MODIS_LST_ID) \
        .filterDate(start_date, end_date) \
        .select('LST_Day_1km')

    def scale_lst(image):
        # 应用缩放因子并将单位从K转换为°C
        return image.multiply(0.02).subtract(273.15).copyProperties(image, ['system:time_start'])

    mean_lst = modis_lst_collection.map(scale_lst).mean().rename('mean_annual_lst_day')

    # 4. 海表温度 (SST) - 20km 缓冲区均值
    # 使用 NOAA OISST V2.1 (2000-2024)
    oisst_collection = ee.ImageCollection("NOAA/CDR/OISST/V2_1") \
        .filterDate(start_date, end_date) \
        .filterBounds(bounds) \
        .select('sst')
    
    # 应用缩放因子: scale=0.01, 结果单位为°C
    def scale_oisst(image):
        return image.multiply(0.01).copyProperties(image, ['system:time_start'])
    
    mean_sst_image = oisst_collection.map(scale_oisst).mean()
    
    # 对 SST 应用 20km 焦点均值滤波 (focal mean)
    # 20000米 / 像素大小(约4600m) ≈ 4.3像素半径,使用5像素确保覆盖
    mean_sst = mean_sst_image.focal_mean(
        radius=20000,
        units='meters',
        kernelType='circle'
    ).rename('mean_annual_sst')

    return ee.Image.cat([mean_temp, min_temp, total_precip, mean_lst, mean_sst])


def add_metadata(feature):
    """
    向每个要素添加经纬度和ID。
    使用原始网格中的 GridID。
    """
    # 获取中心点经纬度
    centroid = feature.geometry().centroid(maxError=1)
    coords = centroid.coordinates()
    
    # 使用原始的 GridID
    grid_id = feature.get('GridID')

    return feature.set({
        'hex_id': grid_id,
        'lon': coords.get(0),
        'lat': coords.get(1)
    })

# --- 主执行流程 ---


print("开始执行主流程...")

# 1. 加载六边形网格
hex_grid = ee.FeatureCollection(HEX_GRID_ASSET_ID)
print(f"成功加载六边形网格，共 {hex_grid.size().getInfo()} 个要素。")

# 2. 计算静态变量
static_vars_image = get_static_variables(COASTLINE_ASSET_ID, JRC_WATER_ID, hex_grid)
# 3. 循环处理每一年
all_years_features = []

for year in TARGET_YEARS:
    print(f"\n正在处理年份: {year}")
    
    # 获取年度变量
    annual_vars_image = get_annual_variables(year, hex_grid)
    
    # 合并所有变量
    combined_image = annual_vars_image.addBands(static_vars_image)
    
    # 按六边形网格提取均值
    print(f"  - 正在为 {year} 年提取数据到网格...")
    reduced_features = combined_image.reduceRegions(
        collection=hex_grid,
        reducer=ee.Reducer.mean(),
        scale=REDUCER_SCALE
    )
    
    # 添加年份和其他元数据
    def set_year_and_meta(feature):
        with_meta = add_metadata(feature)
        return with_meta.set('year', year)
        
    processed_features = reduced_features.map(set_year_and_meta)
    all_years_features.append(processed_features)

# 4. 合并所有年份的结果
print("\n所有年份处理完毕，正在合并结果...")
final_collection = ee.FeatureCollection(ee.List(all_years_features)).flatten()

# 5. 定义导出列
output_properties = [
    'hex_id', 'year', 'lon', 'lat',
    'mean_annual_temp', 'coldest_month_temp', 'total_annual_precip',
    'mean_annual_lst_day', 'mean_annual_sst',
    'dist_to_coast', 'water_occurrence'
]

# 6. 启动导出任务
task = ee.batch.Export.table.toDrive(
    collection=final_collection,
    description=OUTPUT_FILENAME_PREFIX,
    folder=OUTPUT_FOLDER,
    fileNamePrefix=OUTPUT_FILENAME_PREFIX,
    fileFormat='CSV',
    selectors=output_properties
)

task.start()

print("\n" + "="*50)
print("CSV导出任务已成功启动！")
print(f"任务名称: {OUTPUT_FILENAME_PREFIX}")
print(f"文件将保存到您的 Google Drive -> '{OUTPUT_FOLDER}' 文件夹中。")
print("请前往 Google Earth Engine 的 'Tasks' 标签页查看任务进度。")
print("="*50)

配置加载完成。
开始执行主流程...
成功加载六边形网格，共 2512 个要素。
正在计算静态变量 (离岸距离, 水体频率)...

正在处理年份: 2000
  - 开始处理 2000 年的年度变量...
  - 正在为 2000 年提取数据到网格...

正在处理年份: 2005
  - 开始处理 2005 年的年度变量...
  - 正在为 2005 年提取数据到网格...

正在处理年份: 2010
  - 开始处理 2010 年的年度变量...
  - 正在为 2010 年提取数据到网格...

正在处理年份: 2015
  - 开始处理 2015 年的年度变量...
  - 正在为 2015 年提取数据到网格...

正在处理年份: 2020
  - 开始处理 2020 年的年度变量...
  - 正在为 2020 年提取数据到网格...

正在处理年份: 2024
  - 开始处理 2024 年的年度变量...
  - 正在为 2024 年提取数据到网格...

所有年份处理完毕，正在合并结果...

CSV导出任务已成功启动！
任务名称: mangrove_hexgrid_environmental_vars
文件将保存到您的 Google Drive -> 'GEE_Mangrove_Data' 文件夹中。
请前往 Google Earth Engine 的 'Tasks' 标签页查看任务进度。
